# **📧 Email Spam Classification using Trigram NLP Model**


**Welcome to our mini NLP project!** In this notebook, we build a Spam vs Ham email classifier using Unigram + **Bigram** + **Trigram** features and **Naive Bayes**.
The goal is to detect unwanted or harmful emails with high accuracy.

This project is part of our **University assignment**.

# **👥 Team Members**


* Muhammad Hassan Saboor — 25L-8014
* Ali Hussan Malik — 25L-8007


# **📌 1. Import Libraries**

In [1]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# **📌 2. Load Dataset**

In [2]:
df = pd.read_csv("/kaggle/input/spam-emails/spam.csv", encoding="latin-1")

df = df.rename(columns={"Category": "label", "Message": "text"})
df["label"] = df["label"].map({"ham": 0, "spam": 1})

df.head()

,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


# **📌 3. Quick Dataset Overview**

In [3]:
df.shape

(5572, 2)

In [4]:
df["label"].value_counts()

label
0    4825
1     747
Name: count, dtype: int64

# **📌 4. Clean the Text**

In [5]:
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_text)

df.head()

,label,text,clean_text
0,0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,0,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in a wkly comp to win fa cup final ...
3,0,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...


# **📌 5. Train/Test Split**

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["label"], test_size=0.2, random_state=42
)

# **📌 6. Vectorizer (Unigram + Bigram + Trigram)**

In [7]:
vectorizer = CountVectorizer(
    ngram_range=(1, 3),
    analyzer='word',
    min_df=2,
    stop_words='english'
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

len(vectorizer.get_feature_names_out())

9055

# **📌 7. Train Naive Bayes Classifier**

In [8]:
model = MultinomialNB()
model.fit(X_train_vec, y_train)

MultinomialNB()

# **📌 8. Model Evaluation**

In [9]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9838565022421525

Confusion Matrix:
 [[965   1]
 [ 17 132]]

Classification Report:
               precision    recall  f1-score   support

           0       0.98      1.00      0.99       966
           1       0.99      0.89      0.94       149

    accuracy                           0.98      1115
   macro avg       0.99      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115



# **📌 9. Spam Prediction Function**

In [10]:
def predict_email(text):
    text = clean_text(text)
    vec = vectorizer.transform([text])
    pred = model.predict(vec)[0]
    return "SPAM" if pred == 1 else "HAM"

# **📌 10. Test the Model**

In [11]:
sample = "Congratulations! You won a $1000 gift card. Click here to claim."
predict_email(sample)

'SPAM'

# **Thank You**

Thank you for reviewing our notebook! We enjoyed working on this assignment and learned a lot about text processing, n-grams, and email spam detection.

# **💡 Why Not Only Trigrams?**

Because only trigrams miss many short but important spam signals, while combining unigrams + bigrams + trigrams gives much better accuracy and context.